# ▶️ YouTube Engagement Analytics - Advanced Exploratory Data Analysis (EDA)

This notebook performs exploratory data analysis on YouTube engagement datasets. It includes data loading, missing value imputation, statistical summaries, correlation heatmaps, posting schedule analysis, category performance breakdowns, viral video identification, and dataset export.

## 1. Environment Setup & Data Loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Candidate CSV paths
CSV_PATHS = [
    '../data/processed/powerbi_latest.csv',
    '../data/processed/tableau_latest.csv',
    '../data/raw/youtube_videos.csv'
]

df = None
for path in CSV_PATHS:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"[SUCCESS] Loaded CSV from: {path}")
        break

if df is None:
    # Fallback mock dataset
    np.random.seed(42)
    n = 50
    df = pd.DataFrame({
        'video_id': [f'vid_{i}' for i in range(n)],
        'title': [f'YouTube Video Title {i+1}' for i in range(n)],
        'views': np.random.randint(10000, 5000000, size=n),
        'likes': np.random.randint(500, 250000, size=n),
        'comments': np.random.randint(50, 20000, size=n),
        'duration_seconds': np.random.randint(120, 1800, size=n),
        'publish_day': np.random.choice(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'], size=n),
        'category_name': np.random.choice(['Tech', 'Gaming', 'Entertainment', 'Education', 'Science'], size=n)
    })
    print("[INFO] Generated synthetic EDA dataset.")

df.head()

## 2. Metric Engineering & Calculations

In [ ]:
# Compute Engagement Metrics
df['engagement_rate'] = ((df['likes'] + df['comments']) / df['views'].replace(0, np.nan) * 100).round(2)
df['like_ratio'] = (df['likes'] / df['views'].replace(0, np.nan) * 100).round(2)
df['comment_ratio'] = (df['comments'] / df['views'].replace(0, np.nan) * 100).round(2)
df['duration_minutes'] = (df['duration_seconds'] / 60.0).round(1)

# Identify Viral Videos (Views > 2.5x Mean Views)
mean_views = df['views'].mean()
df['is_viral'] = df['views'] >= (mean_views * 2.5)

print(f"Average Channel Views: {mean_views:,.0f}")
print(f"Viral Threshold (2.5x Avg): {mean_views * 2.5:,.0f}")
print(f"Viral Videos Count: {df['is_viral'].sum()}")

## 3. Correlation Analysis & Heatmap

In [ ]:
# Numerical correlation
num_cols = ['views', 'likes', 'comments', 'duration_seconds', 'engagement_rate', 'like_ratio', 'comment_ratio']
avail_num = [c for c in num_cols if c in df.columns]
corr_matrix = df[avail_num].corr()

print("=== CORRELATION MATRIX ===")
print(corr_matrix.round(3))

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('YouTube Metrics Correlation Matrix')
plt.show()

## 4. Category Performance Analytics

In [ ]:
if 'category_name' in df.columns:
    cat_analysis = df.groupby('category_name').agg(
        video_count=('views', 'count'),
        total_views=('views', 'sum'),
        avg_views=('views', 'mean'),
        avg_engagement=('engagement_rate', 'mean')
    ).reset_index().sort_values(by='total_views', ascending=False)
    
    print("=== CATEGORY ANALYTICS ===")
    print(cat_analysis)
    
    plt.figure(figsize=(9, 4))
    sns.barplot(data=cat_analysis, x='category_name', y='avg_views', palette='viridis')
    plt.title('Average Views by Content Category')
    plt.xlabel('Category')
    plt.ylabel('Avg Views')
    plt.show()

## 5. Posting Schedule & Day of Week Performance

In [ ]:
if 'publish_day' in df.columns:
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_analysis = df.groupby('publish_day').agg(
        uploads=('views', 'count'),
        avg_engagement=('engagement_rate', 'mean'),
        avg_views=('views', 'mean')
    ).reindex(day_order).dropna().reset_index()
    
    print("=== DAY OF WEEK PERFORMANCE ===")
    print(day_analysis)
    
    plt.figure(figsize=(9, 4))
    sns.lineplot(data=day_analysis, x='publish_day', y='avg_engagement', marker='o', color='red')
    plt.title('Average Engagement Rate by Day of Week')
    plt.ylabel('Engagement Rate (%)')
    plt.show()

## 6. Export Cleaned EDA Dataset

In [ ]:
export_path = '../data/processed/youtube_engagement_cleaned.csv'
os.makedirs(os.path.dirname(export_path), exist_ok=True)
df.to_csv(export_path, index=False, encoding='utf-8-sig')
print(f"[EXPORT COMPLETE] Cleaned EDA dataset written to: {export_path}")